# Phase 4: Machine Learning Model Training & Evaluation
### Benchmarking Classifiers on the ICBHI 2017 Respiratory Sound Database

Welcome to the **Model Training & Evaluation** notebook! In this notebook, we take the **307 acoustic features** extracted in Phase 3 and train, benchmark, and evaluate four machine learning classifiers on real lung sound recordings.

> **The Goal in Plain English:**  
> We want to see how well standard machine learning algorithms can listen to a patient's breath and answer two critical clinical questions:
> 1. **4-Class Task**: What specific sound is present? (**Normal**, **Crackle**, **Wheeze**, or **Both**)
> 2. **2-Class (Binary) Task**: Is this breath healthy (**Normal**) or unhealthy (**Adventitious**)?

---

### The 4 Classifiers We Are Comparing:
1. **Random Forest (RF)**: An ensemble of 200 decision trees that vote on the sound class. Robust to noisy data and provides clear feature importances.
2. **LightGBM (LGB)**: High-speed gradient-boosted decision trees. Outstanding at finding subtle patterns in tabular feature data.
3. **Support Vector Machine (SVM)**: Uses a non-linear radial basis function (RBF) kernel to find high-dimensional decision boundaries.
4. **Logistic Regression (LR)**: A classic linear benchmark with L2 regularization to set an interpretable baseline.


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report
from lightgbm import LGBMClassifier

try:
    from IPython.display import display
except ImportError:
    display = print

# Configure clean visual styling
%matplotlib inline
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams['figure.dpi'] = 120
print("[OK] All libraries imported successfully!")


## Step 1: Load the Official ICBHI Feature Benchmark Sets

We load the pre-extracted 307-dimensional acoustic feature matrices created in Phase 3:
* **Training Cohort**: 4,142 respiratory cycles across 79 patients.
* **Testing Cohort**: 2,756 respiratory cycles across 49 patients (official challenge split).


In [ ]:
data_dir = "../data/processed"

X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv"))
meta_train = pd.read_csv(os.path.join(data_dir, "metadata_train.csv"))

X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv"))
meta_test = pd.read_csv(os.path.join(data_dir, "metadata_test.csv"))

with open(os.path.join(data_dir, "feature_names.json"), "r") as f:
    feature_info = json.load(f)

print(f"[OK] Training Set: {X_train.shape[0]:,} cycles x {X_train.shape[1]} features (from {meta_train['patient_id'].nunique()} patients)")
print(f"[OK] Test Set    : {X_test.shape[0]:,} cycles x {X_test.shape[1]} features (from {meta_test['patient_id'].nunique()} patients)")

# Class Breakdown Table
class_labels = {0: "Normal", 1: "Crackle", 2: "Wheeze", 3: "Both"}
train_counts = y_train["class_id"].map(class_labels).value_counts()
test_counts = y_test["class_id"].map(class_labels).value_counts()

summary_df = pd.DataFrame({
    "Train Cycles": train_counts,
    "Train (%)": (train_counts / len(y_train) * 100).round(1),
    "Test Cycles": test_counts,
    "Test (%)": (test_counts / len(y_test) * 100).round(1)
})
display(summary_df)


## Step 2: Patient-Aware Evaluation (Why GroupKFold is Essential)

> **Why Random Splitting is Cheating in Healthcare:**  
> Each recording has multiple breath cycles from the same patient. If we randomly shuffle breaths into training and validation folds, the model will see Breaths 1 to 3 of Patient 107 during training, and Breath 4 during validation.  
> The model will simply memorize **Patient 107's unique stethoscope sound and background noise**, rather than learning what a real wheeze or crackle sounds like!  
>  
> **The Solution: 5-Fold `GroupKFold`**  
> We group by `patient_id`. Every single cycle from a patient is either 100% in training or 100% in validation. This guarantees **zero patient leakage**.


In [ ]:
gkf = GroupKFold(n_splits=5)
patient_ids = meta_train["patient_id"].values

print("Demonstrating 5-Fold Patient Partitioning (Zero Patient Overlap):")
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train["class_id"], groups=patient_ids)):
    train_pts = set(patient_ids[tr_idx])
    val_pts = set(patient_ids[val_idx])
    overlap = train_pts.intersection(val_pts)
    print(f"  Fold {fold+1}: {len(tr_idx):,} train cycles ({len(train_pts)} patients) | "
          f"{len(val_idx):,} val cycles ({len(val_pts)} patients) | Overlap = {len(overlap)} patients")


## Step 3: Understanding the Official ICBHI Challenge Metric

Why standard Accuracy can be misleading in medical machine learning:
* In the ICBHI dataset, **52.8% of breaths are Normal**.
* If a useless "lazy model" simply predicted *Normal* for every single breath, it would achieve **52.8% accuracy**, yet miss 100% of sick patients!
* Therefore, the official challenge defines:
  1. **Specificity ($S_p$)**: Accuracy on Normal breathing cycles ($Class = 0$).
  2. **Sensitivity ($S_e$)**: Average recall across abnormal/adventitious cycles ($Class \in \{1, 2, 3\}$).
  3. **Official ICBHI Score**:
     $$\text{Score} = \frac{S_p + S_e}{2}$$


In [ ]:
def compute_icbhi_metrics(y_true, y_pred, is_binary=False):
    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(divide='ignore', invalid='ignore'):
        recalls = cm.diagonal() / cm.sum(axis=1)
        recalls = np.nan_to_num(recalls, nan=0.0)
    
    sp = float(recalls[0])
    se = float(recalls[1]) if is_binary else float(np.mean(recalls[1:]))
    score = float((sp + se) / 2.0)
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    
    return {
        "specificity": sp,
        "sensitivity": se,
        "icbhi_score": score,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "confusion_matrix": cm
    }

print("[OK] ICBHI Metric Calculator ready!")


## Step 4: Load & Review the Benchmark Results

We load the complete benchmark table generated by `scripts/train_models.py` across all 4 machine learning algorithms on both the **4-Class** and **2-Class (Binary)** tasks.


In [ ]:
results_path = "../reports/model_benchmark_results.csv"
results_df = pd.read_csv(results_path)

# Format nicely for display
formatted_df = results_df.copy()
for col in ["cv_icbhi_score", "icbhi_score", "sensitivity", "specificity", "accuracy", "macro_f1"]:
    formatted_df[col] = (formatted_df[col] * 100).round(2).astype(str) + "%"
formatted_df["train_time_sec"] = formatted_df["train_time_sec"].round(1).astype(str) + "s"

formatted_df.rename(columns={
    "task": "Task",
    "model": "Model",
    "cv_icbhi_score": "5-Fold CV Score",
    "icbhi_score": "Official Test Score",
    "sensitivity": "Sensitivity (Se)",
    "specificity": "Specificity (Sp)",
    "accuracy": "Accuracy",
    "macro_f1": "Macro F1",
    "train_time_sec": "Train Time"
}, inplace=True)

display(formatted_df[["Task", "Model", "Official Test Score", "Sensitivity (Se)", "Specificity (Sp)", "Accuracy", "Macro F1", "5-Fold CV Score", "Train Time"]])


## Step 5: Visualizing Model Performance on the Official Test Cohort

Let's compare how each model performed on the unseen official test set (2,756 cycles across 49 patients).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (task_name, ax) in enumerate([("4-Class Task", axes[0]), ("2-Class (Binary) Task", axes[1])]):
    sub_df = results_df[results_df["task"] == task_name].copy()
    plot_data = sub_df.melt(
        id_vars=["model"],
        value_vars=["icbhi_score", "sensitivity", "specificity", "accuracy"],
        var_name="Metric",
        value_name="Score"
    )
    plot_data["Score"] = plot_data["Score"] * 100
    metric_labels = {
        "icbhi_score": "ICBHI Score",
        "sensitivity": "Sensitivity (Se)",
        "specificity": "Specificity (Sp)",
        "accuracy": "Accuracy"
    }
    plot_data["Metric"] = plot_data["Metric"].map(metric_labels)

    palette = {"ICBHI Score": "#1971c2", "Sensitivity (Se)": "#d9480f", "Specificity (Sp)": "#2b8a3e", "Accuracy": "#7048e8"}
    sns.barplot(data=plot_data, x="model", y="Score", hue="Metric", palette=palette, ax=ax, edgecolor="black", linewidth=0.7)
    ax.set_title(f"{task_name} Performance on Official Test Set", fontsize=12, fontweight="bold")
    ax.set_xlabel("Model", fontweight="bold")
    ax.set_ylabel("Score (%)", fontweight="bold")
    ax.set_ylim(0, 100)
    ax.legend(loc="lower right", frameon=True)

fig.suptitle("ICBHI 2017 Benchmark Evaluation Across Machine Learning Baselines", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()


## Step 6: Confusion Matrix Inspection (Where Do Errors Occur?)

Examining normalized confusion matrices on the unseen test set:
* **Left**: 4-Class Classification using Random Forest.
* **Right**: Binary Classification (Normal vs. Adventitious) using LightGBM.

> **Clinical Observations:**  
> 1. **Crackles vs. Normal**: Crackles are brief clicks (transients under 20ms). Without temporal attention, some crackles get misclassified as normal breathing.
> 2. **Wheezes vs. Crackles**: Wheezes are musical continuous tones, so the models can separate them well from crackles, but severe cycles with **Both** crackles and wheezes are rare (only 7.3% of the data) and most difficult to isolate.
> 3. **Binary Performance**: When simplified to *Normal vs. Abnormal*, LightGBM reaches **55.70% ICBHI Score** with balanced sensitivity (59.22%) and specificity (52.18%).


In [ ]:
# Load best trained models from models/ directory
models_dir = "../models"
rf_4c = joblib.load(os.path.join(models_dir, "random_forest_4class.joblib"))
lgb_bin = joblib.load(os.path.join(models_dir, "lightgbm_binary.joblib"))

# Generate predictions on official test set
preds_4c = rf_4c.predict(X_test.values)
preds_bin = lgb_bin.predict(X_test.values)

cm_4c = confusion_matrix(y_test["class_id"], preds_4c, normalize='true') * 100
cm_bin = confusion_matrix(y_test["is_adventitious"], preds_bin, normalize='true') * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

class_labels_4c = ["Normal", "Crackle", "Wheeze", "Both"]
sns.heatmap(cm_4c, annot=True, fmt=".1f", cmap="Blues", cbar=True,
            xticklabels=class_labels_4c, yticklabels=class_labels_4c, ax=axes[0],
            linewidths=0.5, linecolor="gray", annot_kws={"size": 11, "fontweight": "bold"})
axes[0].set_title("Random Forest: 4-Class Normalized Confusion Matrix (%)", fontsize=12, fontweight="bold", pad=10)
axes[0].set_xlabel("Predicted Class", fontweight="bold")
axes[0].set_ylabel("True Clinical Label", fontweight="bold")

class_labels_bin = ["Normal", "Adventitious"]
sns.heatmap(cm_bin, annot=True, fmt=".1f", cmap="Oranges", cbar=True,
            xticklabels=class_labels_bin, yticklabels=class_labels_bin, ax=axes[1],
            linewidths=0.5, linecolor="gray", annot_kws={"size": 11, "fontweight": "bold"})
axes[1].set_title("LightGBM: Binary Normalized Confusion Matrix (%)", fontsize=12, fontweight="bold", pad=10)
axes[1].set_xlabel("Predicted Class", fontweight="bold")
axes[1].set_ylabel("True Clinical Label", fontweight="bold")

fig.suptitle("Normalized Confusion Matrices on Official ICBHI Test Cohort (N = 2,756 cycles)", fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()


## Step 7: Top 20 Most Predictive Acoustic Biomarkers

Which of the 307 acoustic features gave the models the most predictive power?  
We examine the **Mean Decrease in Impurity (Feature Importance)** from the Random Forest model:


In [ ]:
importances = rf_4c.feature_importances_
feature_names = list(X_train.columns)

feat_df = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values(by="Importance", ascending=False).reset_index(drop=True)
top20 = feat_df.head(20)

plt.figure(figsize=(10, 7))
ax = sns.barplot(data=top20, x="Importance", y="Feature", hue="Feature", palette="viridis", edgecolor="black", linewidth=0.6, legend=False)
plt.title("Top 20 Most Predictive Acoustic Biomarkers (Random Forest 4-Class)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Mean Decrease in Impurity (Feature Importance)", fontweight="bold")
plt.ylabel("Acoustic Biomarker", fontweight="bold")
plt.tight_layout()
plt.show()


## Step 8: Clinical Summary & Takeaways

### What Did We Learn from Classical Machine Learning?
1. **LightGBM is the Best Classical Model**:
   * **4-Class Task**: **42.85% ICBHI Score** (55.98% Sp, 29.72% Se, 47.71% Accuracy).
   * **2-Class (Binary) Task**: **55.70% ICBHI Score** (52.18% Sp, 59.22% Se, 55.19% Accuracy).
2. **Acoustic Biomarkers That Matter Most**:
   * **MFCC Static & Delta coefficients** (timbre and rate of spectral change) represent the majority of top biomarkers.
   * **Spectral Centroid, Rolloff, and Bandwidth** directly correspond to the higher frequencies produced by wheezing airways.
   * **Zero-Crossing Rate & RMS Energy** provide key clues for sudden crackle clicks.
3. **The Ceilings of Classical Tabular Features**:
   * Collapsing a dynamic 2.5-second audio breath into static statistical averages (mean, std, skew) discards fine-grained temporal information (e.g. *where* exactly did the crackle click happen within inhalation vs. exhalation?).
   * This is why literature benchmarks show that classical models typically cap out at ~42-45% on 4-Class and ~55-58% on Binary.

---

### Next Step: Phase 5 (Deep Learning)
In Phase 5, we will train **Convolutional Neural Networks (CNNs)** and **Vision Transformers (AST / ResNet)** directly on **Mel-Spectrogram images**, allowing the network to preserve 2D time-frequency structures and push detection accuracy even higher!
